# ML Meta Ensembles — final version

Этот ноутбук строит **ансамбли из meta-tuned ML-моделей** (baseline + cluster + logreg мета-источники),
параметры которых подобраны в `ML_meta_tuning.ipynb` через Optuna.

Паттерн полностью повторяет `ML_ensembles.ipynb`:
- **Single** — одиночные модели как baseline
- **Aggregate** — mean, median, trimmed mean
- **Weighted** — inverse MAE, NNLS, simplex, softmax, rank power
- **Pair-Grid** — grid search на парах
- **Stacker** — мета-обучение (12 алгоритмов)

**Ключевое отличие:** каждая модель использует свой оптимальный мета-источник
(baseline / cluster / logreg), поэтому rebuild-шаг включает аугментацию фичей.

In [ ]:
import os
import sys
import gc
import ast
import json
import math
import random
import shutil
from datetime import datetime
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from scipy.optimize import nnls, minimize
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    BayesianRidge,
    HuberRegressor,
    QuantileRegressor,
    LogisticRegression,
)
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor,
    BaggingRegressor,
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cluster import (
    MiniBatchKMeans, AgglomerativeClustering, Birch,
    DBSCAN, OPTICS, SpectralClustering,
)
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, median_absolute_error,
    pairwise_distances,
)

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

# ── Globals ──
SEED = 2
DURATION_CAP = 960
TARGET_COL = "duration_hours"

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
MOUNT_GOOGLE_DRIVE = True

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / "diploma_ml_runs"
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data"
DRIVE_ARTIFACTS_DIR = DRIVE_PROJECT_DIR / "artifacts_meta_ensembles_final"

if IN_COLAB and USE_GOOGLE_DRIVE and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Google Drive saving disabled or not running in Colab.")

# ── Paths to meta-tuning summary ──
CUSTOM_META_SUMMARY_PATH = "./artifacts_meta_all_baseline_bigsearch_optuna/05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv"
META_SUMMARY_CANDIDATES = [
    Path(CUSTOM_META_SUMMARY_PATH) if CUSTOM_META_SUMMARY_PATH else None,
    Path("./05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv"),
    Path("/mnt/data/05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv"),
    Path("/content/05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv"),
]
if IN_COLAB and USE_GOOGLE_DRIVE:
    META_SUMMARY_CANDIDATES.extend([
        DRIVE_PROJECT_DIR / "05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv",
        DRIVE_PROJECT_DIR / "artifacts_meta_all_baseline_bigsearch_optuna" / "05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv",
    ])
META_SUMMARY_CANDIDATES = [p for p in META_SUMMARY_CANDIDATES if p is not None]

META_SUMMARY_PATH = next((p for p in META_SUMMARY_CANDIDATES if p.exists()), None)
if META_SUMMARY_PATH is None:
    raise FileNotFoundError("Не найден 05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv")

# ── Paths to screening CSVs (needed for cluster/logreg augmentation) ──
LOGREG_SCREEN_CANDIDATES = [
    Path("./artifacts_meta_all_baseline_bigsearch_optuna/03_logreg_screening_all_baseline_bigsearch_optuna.csv"),
    Path("./03_logreg_screening_all_baseline_bigsearch_optuna.csv"),
    Path("/mnt/data/03_logreg_screening_all_baseline_bigsearch_optuna.csv"),
]
CLUSTER_SCREEN_CANDIDATES = [
    Path("./artifacts_meta_all_baseline_bigsearch_optuna/04_cluster_screening_all_baseline_bigsearch_optuna.csv"),
    Path("./04_cluster_screening_all_baseline_bigsearch_optuna.csv"),
    Path("/mnt/data/04_cluster_screening_all_baseline_bigsearch_optuna.csv"),
]

# ── Data paths ──
CUSTOM_DATA_PATH = None
DATA_CANDIDATES = [
    Path(CUSTOM_DATA_PATH) if CUSTOM_DATA_PATH else None,
    Path("./data_finall_without_TTM.csv"),
    Path("/content/data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
    Path("./data_finall.csv"),
    Path("/content/data_finall.csv"),
    Path("/mnt/data/data_finall.csv"),
]
if IN_COLAB and USE_GOOGLE_DRIVE:
    DATA_CANDIDATES.extend([
        DRIVE_DATA_DIR / "data_finall_without_TTM.csv",
        DRIVE_DATA_DIR / "data_finall.csv",
    ])
DATA_CANDIDATES = [p for p in DATA_CANDIDATES if p is not None]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Не найден data_finall_without_TTM.csv / data_finall.csv")

# ── Run config ──
RUN_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
BLEND_FIT_FRAC = 0.50
FORCE_REBUILD_BASE_PREDICTIONS = False
RESUME_IF_PREDICTIONS_EXIST = True

RUN_ALL_PAIRS = True
RUN_ALL_TRIPLES = True
RUN_TOPK_PREFIX_ENSEMBLES = True
RUN_GREEDY_SEARCH = True
RUN_STACKERS = True
RUN_EXHAUSTIVE_TOPN_SUBSETS = True
EXHAUSTIVE_TOP_N = 7
REFIT_TOP_ENSEMBLES = 250
PAIR_WEIGHT_GRID = np.linspace(0.0, 1.0, 101)

# ── Logreg / Cluster meta-feature config (from meta_tuning) ──
USE_PCA = True
PCA_COMPONENTS = 30
LOGREG_OOF_SPLITS = 5

ART_DIR = DRIVE_ARTIFACTS_DIR if (IN_COLAB and USE_GOOGLE_DRIVE) else Path("./artifacts_meta_ensembles_final")
ART_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = ART_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

PRED_CACHE_DIR = RUN_DIR / "pred_cache"
for sub in ["split_val", "split_test_typical", "split_test_full", "fullfit_test_typical", "fullfit_test_full"]:
    (PRED_CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("ART_DIR:", ART_DIR)
print("RUN_DIR:", RUN_DIR)
print("DATA_PATH:", DATA_PATH)
print("META_SUMMARY_PATH:", META_SUMMARY_PATH)


In [ ]:
# ── Seed & serialization helpers ──

def seed_everything(seed: int = 2):
    random.seed(seed)
    np.random.seed(seed)

def serialize_value(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, Path):
        return str(v)
    if pd.isna(v) and not isinstance(v, (str, bytes, bool)):
        return None
    return v

def serialize_params(obj):
    if isinstance(obj, dict):
        return {str(k): serialize_params(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [serialize_params(v) for v in obj]
    return serialize_value(obj)

def json_ready(obj):
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [json_ready(v) for v in obj.tolist()]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    return obj

def sanitize_name(name: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in name)

def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def mae(y, p):
    return float(mean_absolute_error(y, clip_pred(p)))

def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, clip_pred(p))))

def mdae(y, p):
    return float(median_absolute_error(y, clip_pred(p)))

def calc_reg_metrics(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred), "MdAE": mdae(y_true, y_pred)}

def score_predictions(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred), "MdAE": mdae(y_true, y_pred)}

def cleanup_memory():
    gc.collect()

# ── Normalized entropy / margin (for logreg/cluster meta-features) ──

def normalized_entropy(proba):
    proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.zeros(len(proba), dtype=float)
    ent = -(proba * np.log(proba)).sum(axis=1)
    return ent / np.log(proba.shape[1])

def top2_margin(proba):
    proba = np.asarray(proba, dtype=float)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.ones(len(proba), dtype=float)
    desc = np.sort(proba, axis=1)[:, ::-1]
    return desc[:, 0] - desc[:, 1]

# ══════════════════════════════════════════════════════════════
#  LOGREG meta-feature augmentation (from ML_meta_tuning)
# ══════════════════════════════════════════════════════════════

BINNING_SCHEMES = {
    "2cl: 0-200 / 200+": {"mode": "fixed", "edges": [0, 200, np.inf], "labels": ["0-200h", "200+h"]},
    "2cl: 0-500 / 500+": {"mode": "fixed", "edges": [0, 500, np.inf], "labels": ["0-500h", "500+h"]},
    "2cl: median(train)": {"mode": "median", "labels": None},
    "3cl: 0-150 / 151-700 / 700+": {"mode": "fixed", "edges": [0, 150, 700, np.inf], "labels": ["0-150h", "151-700h", "700+h"]},
    "3cl: 0-100 / 101-500 / 500+": {"mode": "fixed", "edges": [0, 100, 500, np.inf], "labels": ["0-100h", "101-500h", "500+h"]},
    "3cl: 0-200 / 201-1000 / 1000+": {"mode": "fixed", "edges": [0, 200, 1000, np.inf], "labels": ["0-200h", "201-1000h", "1000+h"]},
    "4cl: 0-100 / 101-500 / 501-1500 / 1500+": {"mode": "fixed", "edges": [0, 100, 500, 1500, np.inf], "labels": ["0-100h", "101-500h", "501-1500h", "1500+h"]},
    "4cl: 0-200 / 201-700 / 701-2000 / 2000+": {"mode": "fixed", "edges": [0, 200, 700, 2000, np.inf], "labels": ["0-200h", "201-700h", "701-2000h", "2000+h"]},
    "4cl: quantile_4(train)": {"mode": "quantile", "q": 4, "labels": None},
    "5cl: 0-200 / 201-1000 / 1001-2000 / 2001-3000 / 3000+": {"mode": "fixed", "edges": [0, 200, 1000, 2000, 3000, np.inf], "labels": ["0-200h", "201-1000h", "1001-2000h", "2001-3000h", "3000+h"]},
    "5cl: 0-50 / 51-200 / 201-500 / 501-1500 / 1500+": {"mode": "fixed", "edges": [0, 50, 200, 500, 1500, np.inf], "labels": ["0-50h", "51-200h", "201-500h", "501-1500h", "1500+h"]},
    "5cl: quantile_5(train)": {"mode": "quantile", "q": 5, "labels": None},
    "6cl: 0-24 / 25-100 / 101-300 / 301-700 / 701-2000 / 2000+": {"mode": "fixed", "edges": [0, 24, 100, 300, 700, 2000, np.inf], "labels": ["0-24h", "25-100h", "101-300h", "301-700h", "701-2000h", "2000+h"]},
}

def fit_binning_on_train(y_train, scheme_cfg):
    mode = scheme_cfg["mode"]
    if mode == "fixed":
        return {"edges": list(scheme_cfg["edges"]), "labels": list(scheme_cfg["labels"])}
    if mode == "median":
        med = float(y_train.median())
        return {"edges": [float(y_train.min()) - 1e-9, med, np.inf], "labels": [f"<= {med:.0f}", f"> {med:.0f}"]}
    if mode == "quantile":
        _, edges = pd.qcut(y_train, q=scheme_cfg["q"], retbins=True, duplicates="drop")
        edges = list(edges)
        edges[0] = float(min(edges[0], y_train.min() - 1e-9))
        edges[-1] = np.inf
        labels = [f"class_{i}" for i in range(len(edges) - 1)]
        return {"edges": edges, "labels": labels}
    raise ValueError(mode)

def apply_binning(y, fitted_binning):
    cls = pd.cut(y, bins=fitted_binning["edges"], labels=False, include_lowest=True, right=True)
    if cls.isna().any():
        raise RuntimeError(f"NaN after binning: {int(cls.isna().sum())}")
    return cls.astype(int).values

def compute_soft_class_weight(y_cls, alpha):
    counts = pd.Series(y_cls).value_counts().sort_index()
    n, k = len(y_cls), len(counts)
    weights = {}
    for c, cnt in counts.items():
        balanced = n / (k * cnt)
        weights[int(c)] = float(1.0 + alpha * (balanced - 1.0))
    return weights

def build_logreg_classifier(alpha, y_cls):
    class_weight = compute_soft_class_weight(y_cls, alpha)
    return Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(random_state=SEED, max_iter=4000, solver="lbfgs", class_weight=class_weight))])

def aligned_proba(clf, X, n_classes):
    raw = clf.predict_proba(X)
    out = np.zeros((len(X), n_classes), dtype=float)
    seen_classes = clf.named_steps["m"].classes_
    for i, cls in enumerate(seen_classes):
        out[:, int(cls)] = raw[:, i]
    return out

def empirical_prior_from_past(y_past, n_classes):
    if len(y_past) == 0:
        return np.ones(n_classes, dtype=float) / n_classes
    counts = np.bincount(np.asarray(y_past, dtype=int), minlength=n_classes).astype(float)
    return counts / counts.sum() if counts.sum() > 0 else np.ones(n_classes, dtype=float) / n_classes

def make_time_oof_logreg_features(X_train, y_train_cls, alpha, n_splits=LOGREG_OOF_SPLITS):
    X_train = X_train.reset_index(drop=True)
    y_train_cls = np.asarray(y_train_cls, dtype=int)
    n = len(X_train)
    n_classes = int(np.max(y_train_cls)) + 1
    if n < 4 or n_classes < 2:
        prior = empirical_prior_from_past([], n_classes)
        proba = np.tile(prior, (n, 1))
        return np.argmax(proba, axis=1), proba
    n_splits = max(2, min(n_splits, n - 1))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    oof_proba = np.zeros((n, n_classes), dtype=float)
    covered = np.zeros(n, dtype=bool)
    for tr_idx, va_idx in tscv.split(X_train):
        ytr = y_train_cls[tr_idx]
        prior = empirical_prior_from_past(ytr, n_classes)
        if len(np.unique(ytr)) < 2:
            oof_proba[va_idx] = np.tile(prior, (len(va_idx), 1))
            covered[va_idx] = True
            continue
        clf = build_logreg_classifier(alpha, ytr)
        clf.fit(X_train.iloc[tr_idx], ytr)
        oof_proba[va_idx] = aligned_proba(clf, X_train.iloc[va_idx], n_classes)
        covered[va_idx] = True
    missing_idx = np.where(~covered)[0]
    for idx in missing_idx:
        oof_proba[idx] = empirical_prior_from_past(y_train_cls[:idx], n_classes)
    return np.argmax(oof_proba, axis=1), oof_proba

def make_logreg_meta_frame(pred_cls, pred_proba, prefix="logreg"):
    return pd.DataFrame({
        f"{prefix}_pred_class": np.asarray(pred_cls, dtype=int),
        f"{prefix}_max_proba": np.max(pred_proba, axis=1),
        f"{prefix}_entropy": normalized_entropy(pred_proba),
        f"{prefix}_margin_top1_top2": top2_margin(pred_proba),
    })

def build_logreg_augmented(train_df, infer_df, scheme_name, alpha):
    train_df = train_df.reset_index(drop=True)
    infer_df = infer_df.reset_index(drop=True)
    X_train = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    y_train = train_df[TARGET_COL].reset_index(drop=True)
    X_infer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    fitted_binning = fit_binning_on_train(y_train, BINNING_SCHEMES[scheme_name])
    y_cls = apply_binning(y_train, fitted_binning)
    n_classes = int(np.max(y_cls)) + 1
    if n_classes < 2:
        return None
    train_pred_cls, train_proba = make_time_oof_logreg_features(X_train, y_cls, alpha)
    if len(np.unique(y_cls)) < 2:
        infer_proba = np.tile(empirical_prior_from_past(y_cls, n_classes), (len(X_infer), 1))
    else:
        clf_full = build_logreg_classifier(alpha, y_cls)
        clf_full.fit(X_train, y_cls)
        infer_proba = aligned_proba(clf_full, X_infer, n_classes)
    infer_pred_cls = np.argmax(infer_proba, axis=1)
    train_meta = make_logreg_meta_frame(train_pred_cls, train_proba)
    infer_meta = make_logreg_meta_frame(infer_pred_cls, infer_proba)
    X_train_aug = pd.concat([X_train, train_meta], axis=1)
    X_infer_aug = pd.concat([X_infer, infer_meta], axis=1)
    meta_info = {"scheme": scheme_name, "alpha": float(alpha), "n_classes": int(n_classes)}
    return X_train_aug, X_infer_aug, meta_info

# ══════════════════════════════════════════════════════════════
#  CLUSTER meta-feature augmentation (from ML_meta_tuning)
# ══════════════════════════════════════════════════════════════

rng = np.random.default_rng(SEED)
CLUSTER_MIN_VALID_CLUSTERS = 2
CLUSTER_MAX_VALID_CLUSTERS = 60
CLUSTER_MAX_HEAVY_FIT_ROWS = 3000

def stable_softmax_negdist(dist):
    dist = np.asarray(dist, dtype=float)
    scale = np.std(dist) if np.std(dist) > 1e-12 else 1.0
    z = -dist / scale
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def probs_from_dist(all_d):
    return stable_softmax_negdist(all_d)

def fit_preprocessor(X_train):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)
    pca = None
    Xt = Xs
    if USE_PCA and Xs.shape[1] > PCA_COMPONENTS:
        pca = PCA(n_components=PCA_COMPONENTS, random_state=SEED)
        Xt = pca.fit_transform(Xs)
    return scaler, pca, Xt

def apply_preprocessor(X, scaler, pca):
    Xs = scaler.transform(X)
    return pca.transform(Xs) if pca is not None else Xs

def build_centroids(X, labels):
    labels = np.asarray(labels, dtype=int)
    valid_mask = labels != -1
    valid_labels = np.unique(labels[valid_mask])
    if len(valid_labels) < CLUSTER_MIN_VALID_CLUSTERS:
        return None, None
    label_map = {old: new for new, old in enumerate(sorted(valid_labels))}
    mapped = np.full_like(labels, -1)
    centroids = []
    for old_label in sorted(valid_labels):
        m = labels == old_label
        mapped[m] = label_map[old_label]
        centroids.append(X[m].mean(axis=0))
    centroids = np.vstack(centroids)
    if len(centroids) > CLUSTER_MAX_VALID_CLUSTERS:
        return None, None
    return centroids, mapped

def assign_by_centroids(X, centroids):
    all_d = pairwise_distances(X, centroids, metric="euclidean")
    labels = np.argmin(all_d, axis=1)
    dmin = all_d[np.arange(len(X)), labels]
    return labels, dmin, all_d

def make_clusterer(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(n_clusters=params["n_clusters"], random_state=SEED, n_init=20, batch_size=1024)
    elif name == "GaussianMixture":
        return GaussianMixture(n_components=params["n_components"], covariance_type=params["covariance_type"], random_state=SEED)
    elif name == "Ward":
        return AgglomerativeClustering(n_clusters=params["n_clusters"], linkage="ward")
    elif name == "AgglomerativeClustering":
        return AgglomerativeClustering(n_clusters=params["n_clusters"], linkage=params["linkage"])
    elif name == "DBSCAN":
        return DBSCAN(eps=params["eps"], min_samples=params["min_samples"])
    elif name in {"BIRCH", "Birch"}:
        return Birch(n_clusters=params["n_clusters"], threshold=params["threshold"])
    elif name == "SpectralClustering":
        return SpectralClustering(n_clusters=params["n_clusters"], affinity=params.get("affinity", "nearest_neighbors"), random_state=SEED, assign_labels="kmeans")
    elif name == "OPTICS":
        return OPTICS(min_samples=params["min_samples"], xi=params["xi"])
    elif name == "HDBSCAN":
        return hdbscan.HDBSCAN(min_cluster_size=params["min_cluster_size"], min_samples=params["min_samples"], prediction_data=True)
    else:
        raise ValueError(name)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    Xfit = Xtr
    fit_mode = "full_train"
    if name in {"SpectralClustering", "HDBSCAN", "OPTICS", "DBSCAN"} and len(Xtr) > CLUSTER_MAX_HEAVY_FIT_ROWS:
        idx = rng.choice(len(Xtr), size=CLUSTER_MAX_HEAVY_FIT_ROWS, replace=False)
        idx = np.sort(idx)
        Xfit = Xtr[idx]
        fit_mode = f"subsample_{CLUSTER_MAX_HEAVY_FIT_ROWS}"

    est = make_clusterer(name, params)

    if name == "GaussianMixture":
        est.fit(Xfit)
        tr_labels = est.predict(Xtr)
        te_labels = est.predict(Xte)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)
        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    if name == "HDBSCAN":
        est.fit(Xfit)
        fit_labels = est.labels_
        if len(np.unique(fit_labels[fit_labels != -1])) < CLUSTER_MIN_VALID_CLUSTERS:
            return None
        try:
            te_labels, _ = hdbscan.approximate_predict(est, Xte)
        except Exception:
            return None
        centroids, _ = build_centroids(Xfit, fit_labels)
        if centroids is None:
            return None
        tr_labels, tr_dmin, tr_d = assign_by_centroids(Xtr, centroids)
        te_labels, te_dmin, te_d = assign_by_centroids(Xte, centroids)
        tr_proba = probs_from_dist(tr_d)
        te_proba = probs_from_dist(te_d)
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native_test"

    if hasattr(est, "fit_predict"):
        fit_labels = est.fit_predict(Xfit)
    else:
        est.fit(Xfit)
        fit_labels = est.labels_ if hasattr(est, "labels_") else est.predict(Xfit)

    if hasattr(est, "predict") and name in {"MiniBatchKMeans", "BIRCH", "Birch"} and len(Xfit) == len(Xtr):
        tr_labels = est.predict(Xtr)
        te_labels = est.predict(Xte)
        if hasattr(est, "cluster_centers_"):
            tr_d = pairwise_distances(Xtr, est.cluster_centers_)
            te_d = pairwise_distances(Xte, est.cluster_centers_)
            tr_proba = probs_from_dist(tr_d)
            te_proba = probs_from_dist(te_d)
        else:
            tr_proba = te_proba = None
        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    centroids, _ = build_centroids(Xfit, fit_labels)
    if centroids is None:
        return None
    tr_labels, tr_dmin, tr_d = assign_by_centroids(Xtr, centroids)
    te_labels, te_dmin, te_d = assign_by_centroids(Xte, centroids)
    tr_proba = probs_from_dist(tr_d)
    te_proba = probs_from_dist(te_d)
    if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
        return None
    return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_centroid"

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})
    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)
    tr_feat["cluster_is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["cluster_is_noise"] = (te_feat["cluster_id"] == -1).astype(int)
    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster")
    te_ohe = te_ohe.reindex(columns=tr_ohe.columns, fill_value=0)
    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)
    if tr_proba is not None and te_proba is not None:
        tr_feat["cluster_max_proba"] = np.max(tr_proba, axis=1)
        te_feat["cluster_max_proba"] = np.max(te_proba, axis=1)
        tr_feat["cluster_entropy"] = normalized_entropy(tr_proba)
        te_feat["cluster_entropy"] = normalized_entropy(te_proba)
        tr_feat["cluster_margin_top1_top2"] = top2_margin(tr_proba)
        te_feat["cluster_margin_top1_top2"] = top2_margin(te_proba)
        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]
    else:
        tr_feat["cluster_max_proba"] = 1.0
        te_feat["cluster_max_proba"] = 1.0
        tr_feat["cluster_entropy"] = 0.0
        te_feat["cluster_entropy"] = 0.0
        tr_feat["cluster_margin_top1_top2"] = 1.0
        te_feat["cluster_margin_top1_top2"] = 1.0
    return tr_feat, te_feat

def build_cluster_augmented(train_df, infer_df, cluster_name, cluster_params):
    train_df = train_df.reset_index(drop=True)
    infer_df = infer_df.reset_index(drop=True)
    X_train = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    X_infer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    scaler, pca, Xt_train = fit_preprocessor(X_train)
    Xt_infer = apply_preprocessor(X_infer, scaler, pca)
    result = fit_clusterer_and_assign(cluster_name, cluster_params, Xt_train, Xt_infer)
    if result is None:
        return None
    tr_labels, te_labels, tr_proba, te_proba, fit_mode = result
    tr_feat, te_feat = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)
    X_train_aug = pd.concat([X_train, tr_feat.reset_index(drop=True)], axis=1)
    X_infer_aug = pd.concat([X_infer, te_feat.reset_index(drop=True)], axis=1)
    meta_info = {"cluster_name": cluster_name, "cluster_params": json_ready(cluster_params), "fit_mode": fit_mode}
    return X_train_aug, X_infer_aug, meta_info

# ══════════════════════════════════════════════════════════════
#  Source augmentation dispatcher (from ML_meta_tuning)
# ══════════════════════════════════════════════════════════════

def build_source_augmented_data(meta_source, config, train_df, infer_df):
    if meta_source == "baseline":
        Xtr = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
        Xinfer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
        return Xtr, Xinfer, {"meta_source": "baseline"}
    if meta_source == "logreg":
        return build_logreg_augmented(train_df, infer_df, config["scheme"], config["alpha"])
    if meta_source == "cluster":
        return build_cluster_augmented(train_df, infer_df, config["cluster_name"], config["cluster_params"])
    raise ValueError(meta_source)

# ══════════════════════════════════════════════════════════════
#  BASELINE PIPELINES (from ML_meta_tuning)
# ══════════════════════════════════════════════════════════════

BASELINE_PIPELINES = {
    "Scaled_Lasso": lambda: Pipeline([("Scaler", StandardScaler()), ("Lasso", Lasso(random_state=SEED))]),
    "Scaled_Elastic": lambda: Pipeline([("Scaler", StandardScaler()), ("Elastic", ElasticNet(random_state=SEED))]),
    "Scaled_RF_reg": lambda: Pipeline([("Scaler", StandardScaler()), ("RF", RandomForestRegressor(random_state=SEED))]),
    "Scaled_ET_reg": lambda: Pipeline([("Scaler", StandardScaler()), ("ET", ExtraTreesRegressor(random_state=SEED))]),
    "Scaled_BR_reg": lambda: Pipeline([("Scaler", StandardScaler()), ("BR", BaggingRegressor(random_state=SEED))]),
    "Scaled_DT_reg": lambda: Pipeline([("Scaler", StandardScaler()), ("DT_reg", DecisionTreeRegressor(random_state=SEED))]),
    "Scaled_Ridge": lambda: Pipeline([("Scaler", StandardScaler()), ("Ridge", Ridge(random_state=SEED))]),
    "Scaled_SVR": lambda: Pipeline([("Scaler", StandardScaler()), ("SVR", SVR(kernel="linear", C=1e2))]),
    "Scaled_Hub-Reg": lambda: Pipeline([("Scaler", StandardScaler()), ("Hub-Reg", HuberRegressor())]),
    "Scaled_BayRidge": lambda: Pipeline([("Scaler", StandardScaler()), ("BR", BayesianRidge())]),
    "Scaled_KNN_reg": lambda: Pipeline([("Scaler", StandardScaler()), ("KNN_reg", KNeighborsRegressor())]),
    "Scaled_Gboost-Reg": lambda: Pipeline([("Scaler", StandardScaler()), ("GBoost-Reg", GradientBoostingRegressor(random_state=SEED))]),
    "Scaled_XGB_reg": lambda: Pipeline([("Scaler", StandardScaler()), ("XGBR", XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0))]),
    "Scaled_RFR_PCA": lambda: Pipeline([("Scaler", StandardScaler()), ("PCA", PCA(n_components=3, random_state=SEED)), ("RF", RandomForestRegressor(random_state=SEED))]),
    "Scaled_XGBR_PCA": lambda: Pipeline([("Scaler", StandardScaler()), ("PCA", PCA(n_components=3, random_state=SEED)), ("XGB", XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0))]),
}

def build_regression_model(model_name, extra_params=None):
    if model_name not in BASELINE_PIPELINES:
        raise ValueError(f"Unknown model: {model_name}")
    est = BASELINE_PIPELINES[model_name]()
    if extra_params:
        safe = {}
        for k, v in extra_params.items():
            if "__" in k:
                safe[k] = v
            else:
                # try to attach to the last step
                last_step_name = est.steps[-1][0]
                safe[f"{last_step_name}__{k}"] = v
        est.set_params(**safe)
    return est

# ══════════════════════════════════════════════════════════════
#  ENSEMBLE functions (from ML_ensembles)
# ══════════════════════════════════════════════════════════════

def aggregate_predictions(pred_matrix: np.ndarray, method: str, trim_frac: float = 0.1):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    if pred_matrix.ndim == 1:
        return clip_pred(pred_matrix)
    if method == "mean":
        out = pred_matrix.mean(axis=1)
    elif method == "median":
        out = np.median(pred_matrix, axis=1)
    elif method == "trimmed_mean":
        m = pred_matrix.shape[1]
        k = int(np.floor(m * trim_frac))
        if k <= 0 or 2 * k >= m:
            out = pred_matrix.mean(axis=1)
        else:
            sorted_mat = np.sort(pred_matrix, axis=1)
            out = sorted_mat[:, k:m-k].mean(axis=1)
    else:
        raise KeyError(method)
    return clip_pred(out)

def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    s = w.sum()
    return w / s if s > 0 else np.ones_like(w) / len(w)

def weights_from_errors(pred_fit: pd.DataFrame, y_fit, scheme: str, temperature: float = 1.0, power: float = 1.0):
    errs = np.array([mae(y_fit, pred_fit[c].values) for c in pred_fit.columns], dtype=float)
    if scheme == "equal":
        w = np.ones(len(errs))
    elif scheme == "inverse_mae":
        w = 1.0 / np.maximum(errs, 1e-9)
    elif scheme == "inverse_mae_sq":
        w = 1.0 / np.maximum(errs, 1e-9) ** 2
    elif scheme == "softmax_mae":
        z = -errs / max(float(temperature), 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "rank_power":
        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(errs) + 1)
        w = 1.0 / np.power(ranks, power)
    else:
        raise KeyError(scheme)
    return normalize_weights(w)

def fit_weighted_blender(pred_fit: pd.DataFrame, y_fit, scheme: str, **kwargs):
    X = pred_fit.values.astype(float)
    if scheme in {"equal", "inverse_mae", "inverse_mae_sq", "softmax_mae", "rank_power"}:
        w = weights_from_errors(pred_fit, y_fit, scheme, **kwargs)
        return {"kind": "weights", "weights": w}
    elif scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        return {"kind": "weights", "weights": normalize_weights(w)}
    elif scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
        def objective(w):
            return mae(y_fit, X @ w)
        res = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=[cons], options={"maxiter": 500, "ftol": 1e-8})
        return {"kind": "weights", "weights": normalize_weights(res.x if res.success else x0), "optimizer_success": bool(res.success)}
    else:
        raise KeyError(scheme)

def predict_weighted_blender(fitted, pred_df: pd.DataFrame):
    return clip_pred(pred_df.values @ np.asarray(fitted["weights"], dtype=float))

def build_stacker(stacker_name: str):
    if stacker_name == "linear":        return LinearRegression()
    elif stacker_name == "positive_linear": return LinearRegression(positive=True)
    elif stacker_name == "ridge":       return Ridge(alpha=1.0)
    elif stacker_name == "lasso":       return Lasso(alpha=1e-3, max_iter=20000, random_state=SEED)
    elif stacker_name == "elastic":     return ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=20000, random_state=SEED)
    elif stacker_name == "bayes":       return BayesianRidge()
    elif stacker_name == "huber":       return HuberRegressor(max_iter=2000, epsilon=1.35)
    elif stacker_name == "quantile":    return QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs")
    elif stacker_name == "gbr":         return GradientBoostingRegressor(loss="absolute_error", random_state=SEED, n_estimators=300, learning_rate=0.05, max_depth=2, min_samples_leaf=5, min_samples_split=4, subsample=0.9)
    elif stacker_name == "rf":          return RandomForestRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "etr":         return ExtraTreesRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "xgb":
        if not HAS_XGB:
            raise RuntimeError("xgboost not available")
        return XGBRegressor(objective="reg:absoluteerror", eval_metric="mae", n_estimators=500, learning_rate=0.03, max_depth=2, min_child_weight=3, subsample=0.9, colsample_bytree=0.9, tree_method="hist", n_jobs=-1, random_state=SEED)
    else:
        raise KeyError(stacker_name)

def fit_stacker(pred_fit: pd.DataFrame, y_fit, stacker_name: str):
    model = build_stacker(stacker_name)
    model.fit(pred_fit.values.astype(float), np.asarray(y_fit, dtype=float))
    return {"kind": "stacker", "model": model, "stacker_name": stacker_name}

def predict_stacker(fitted, pred_df: pd.DataFrame):
    return clip_pred(fitted["model"].predict(pred_df.values.astype(float)))

def fit_pair_weight_grid(pred_fit: pd.DataFrame, y_fit):
    if pred_fit.shape[1] != 2:
        raise ValueError("pair_grid expects exactly 2 models")
    best = None
    a = pred_fit.iloc[:, 0].values.astype(float)
    b = pred_fit.iloc[:, 1].values.astype(float)
    for w in PAIR_WEIGHT_GRID:
        pred = clip_pred(w * a + (1.0 - w) * b)
        score = mae(y_fit, pred)
        if best is None or score < best["selection_MAE"]:
            best = {"kind": "pair_grid", "weight_first": float(w), "selection_MAE": score}
    return best

def predict_pair_weight_grid(fitted, pred_df: pd.DataFrame):
    w = float(fitted["weight_first"])
    a = pred_df.iloc[:, 0].values.astype(float)
    b = pred_df.iloc[:, 1].values.astype(float)
    return clip_pred(w * a + (1.0 - w) * b)

# ── Ensemble spec helpers ──

def spec_tag(spec):
    models_part = "__".join(spec["models"])
    return f'{spec["family"]}::{spec["name"]}::{models_part}'

def get_base_leaderboard_from_predictions(pred_df: pd.DataFrame, y_true):
    y_true = np.asarray(y_true, dtype=float)
    rows = []
    for col in pred_df.columns:
        rows.append({"model": col, "MAE": mae(y_true, pred_df[col].values), "RMSE": rmse(y_true, pred_df[col].values), "MdAE": mdae(y_true, pred_df[col].values)})
    return pd.DataFrame(rows).sort_values(["MAE", "RMSE", "MdAE"], kind="stable").reset_index(drop=True)

def evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel):
    models = spec["models"]
    fit_sub = pred_fit_df[models].copy()
    sel_sub = pred_sel_df[models].copy()
    family = spec["family"]
    name = spec["name"]
    if family == "single":
        pred_sel = clip_pred(sel_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_sel = aggregate_predictions(sel_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(fit_sub, y_fit, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_sel = predict_weighted_blender(fitted, sel_sub)
        out = {"family": family, "name": name, "models": models, "n_models": len(models), "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", [])}
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(fit_sub, y_fit)
        pred_sel = predict_pair_weight_grid(fitted, sel_sub)
        out = {"family": family, "name": name, "models": models, "n_models": len(models), "weight_first": fitted["weight_first"]}
    elif family == "stacker":
        fitted = fit_stacker(fit_sub, y_fit, spec["stacker_name"])
        pred_sel = predict_stacker(fitted, sel_sub)
        out = {"family": family, "name": name, "models": models, "n_models": len(models), "stacker_name": spec["stacker_name"]}
    else:
        raise KeyError(family)
    sel_metrics = score_predictions(y_sel, pred_sel)
    out["selection_MAE"] = sel_metrics["MAE"]
    out["selection_RMSE"] = sel_metrics["RMSE"]
    out["selection_MdAE"] = sel_metrics["MdAE"]
    out["tag"] = spec_tag(spec)
    out["spec"] = spec
    return out

def refit_and_eval_spec(spec, pred_val_df, y_val, pred_test_typ_df, y_test_typ, pred_test_full_df, y_test_full):
    models = spec["models"]
    val_sub = pred_val_df[models].copy()
    test_typ_sub = pred_test_typ_df[models].copy()
    test_full_sub = pred_test_full_df[models].copy()
    family = spec["family"]
    name = spec["name"]
    if family == "single":
        pred_typ = clip_pred(test_typ_sub.iloc[:, 0].values)
        pred_full = clip_pred(test_full_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_typ = aggregate_predictions(test_typ_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        pred_full = aggregate_predictions(test_full_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(val_sub, y_val, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_typ = predict_weighted_blender(fitted, test_typ_sub)
        pred_full = predict_weighted_blender(fitted, test_full_sub)
        out = {"family": family, "name": name, "models": models, "n_models": len(models), "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", [])}
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(val_sub, y_val)
        pred_typ = predict_pair_weight_grid(fitted, test_typ_sub)
        pred_full = predict_pair_weight_grid(fitted, test_full_sub)
        out = {"family": family, "name": name, "models": models, "n_models": len(models), "weight_first": fitted["weight_first"], "weights_like": [fitted["weight_first"], 1.0 - fitted["weight_first"]]}
    elif family == "stacker":
        fitted = fit_stacker(val_sub, y_val, spec["stacker_name"])
        pred_typ = predict_stacker(fitted, test_typ_sub)
        pred_full = predict_stacker(fitted, test_full_sub)
        weights_like = None
        if hasattr(fitted["model"], "coef_"):
            coef = np.ravel(getattr(fitted["model"], "coef_"))
            if len(coef) == len(models):
                weights_like = coef.tolist()
        out = {"family": family, "name": name, "models": models, "n_models": len(models), "weights_like": weights_like or []}
    else:
        raise KeyError(family)
    typ_metrics = score_predictions(y_test_typ, pred_typ)
    full_metrics = score_predictions(y_test_full, pred_full)
    out.update({"tag": spec_tag(spec), "MAE_typical": typ_metrics["MAE"], "RMSE_typical": typ_metrics["RMSE"], "MdAE_typical": typ_metrics["MdAE"], "MAE_full": full_metrics["MAE"], "RMSE_full": full_metrics["RMSE"], "MdAE_full": full_metrics["MdAE"]})
    return out

def greedy_forward_specs(ranked_models):
    specs = []
    current = []
    remaining = ranked_models.copy()
    while remaining:
        best_candidate = None
        for m in remaining:
            cand = current + [m]
            specs.append({"family": "aggregate", "name": f"greedy_mean_step{len(cand)}", "models": cand, "agg_method": "mean"})
            specs.append({"family": "weighted", "name": f"greedy_inverse_step{len(cand)}", "models": cand, "weight_scheme": "inverse_mae"})
            best_candidate = m
        current = current + [best_candidate]
        remaining = [m for m in remaining if m != best_candidate]
    return specs

print("All helper functions defined.")


## Load data & meta-tuning summary

Загружаем данные и результаты meta-tuning (best model + best source per model).

In [ ]:
seed_everything(SEED)

# ── Load data ──
data = pd.read_csv(DATA_PATH)
split = int(len(data) * 0.8)

train_full = data.iloc[:split].copy()
test_full = data.iloc[split:].copy()

train_typical = train_full[train_full[TARGET_COL] < DURATION_CAP].copy()
test_typical = test_full[test_full[TARGET_COL] < DURATION_CAP].copy()

# ── Inner temporal split for ensemble blend search ──
blend_split_idx = int(len(train_typical) * 0.85)
blend_train = train_typical.iloc[:blend_split_idx].copy().reset_index(drop=True)
blend_val = train_typical.iloc[blend_split_idx:].copy().reset_index(drop=True)

X_blend_tr_raw = blend_train.drop(columns=[TARGET_COL])
y_blend_tr = blend_train[TARGET_COL].reset_index(drop=True)
X_blend_val_raw = blend_val.drop(columns=[TARGET_COL])
y_blend_val = blend_val[TARGET_COL].reset_index(drop=True)

Xtrain_raw = train_typical.drop(columns=[TARGET_COL])
ytrain = train_typical[TARGET_COL].reset_index(drop=True)
Xtest_typical_raw = test_typical.drop(columns=[TARGET_COL])
ytest_typical = test_typical[TARGET_COL].reset_index(drop=True)
Xtest_full_raw = test_full.drop(columns=[TARGET_COL])
ytest_full = test_full[TARGET_COL].reset_index(drop=True)

print(f"Train raw: {len(train_full)}")
print(f"Train typical < {DURATION_CAP}: {len(train_typical)}")
print(f"Inner blend train: {len(blend_train)}")
print(f"Inner blend val: {len(blend_val)}")
print(f"Holdout typical: {len(test_typical)}")
print(f"Holdout full: {len(test_full)}")

# ── Load meta-tuning best-by-model ──
meta_df = pd.read_csv(META_SUMMARY_PATH)
print(f"\nLoaded meta summary from: {META_SUMMARY_PATH}")
print(f"Models in summary: {len(meta_df)}")
display(meta_df[["model", "meta_source", "source_tag", "cv_MAE"]])

SELECTED_MODELS = meta_df["model"].tolist()
print(f"\nSelected models for ensembling: {len(SELECTED_MODELS)}")


## Rebuild meta-tuned models → prediction matrices

Для каждой модели:
1. Восстанавливаем мета-источник (baseline / cluster / logreg)
2. Аугментируем фичи
3. Собираем Pipeline с оптимальными гиперпараметрами
4. Fit → predict на blend-val и holdout splits

In [ ]:
def _pred_path(kind: str, model_name: str) -> Path:
    return PRED_CACHE_DIR / kind / f"{sanitize_name(model_name)}.npy"

def _save_pred(kind: str, model_name: str, arr):
    np.save(_pred_path(kind, model_name), np.asarray(arr, dtype=np.float32))

def _load_pred(kind: str, model_name: str):
    path = _pred_path(kind, model_name)
    return np.load(path) if path.exists() else None

def _have_all_cached(model_name: str) -> bool:
    needed = [
        _pred_path("split_val", model_name),
        _pred_path("split_test_typical", model_name),
        _pred_path("split_test_full", model_name),
        _pred_path("fullfit_test_typical", model_name),
        _pred_path("fullfit_test_full", model_name),
    ]
    return all(p.exists() for p in needed)

def load_prediction_frame(kind: str, model_names):
    cols = {}
    for m in model_names:
        arr = _load_pred(kind, m)
        if arr is not None:
            cols[m] = arr
    return pd.DataFrame(cols)


def rebuild_meta_model(row):
    """Rebuild one meta-tuned model: augment features + fit + predict."""
    model_name = row["model"]
    meta_source = row["meta_source"]
    source_config = json.loads(row["source_config_json"]) if isinstance(row["source_config_json"], str) else row["source_config_json"]
    extra_params = json.loads(row["best_params_json"]) if isinstance(row["best_params_json"], str) else row["best_params_json"]

    print(f"\n=== Rebuilding {model_name} [source={meta_source}] ===")
    t0 = datetime.now()

    if RESUME_IF_PREDICTIONS_EXIST and not FORCE_REBUILD_BASE_PREDICTIONS and _have_all_cached(model_name):
        print("  Using cached predictions.")
        return {"model": model_name, "status": "cached", "elapsed_sec": 0.0}

    seed_everything(SEED)

    # ── 1) Split-fit: train on blend_train, predict on blend_val ──
    try:
        res_split_tr = build_source_augmented_data(meta_source, source_config, blend_train, blend_val)
    except Exception as e:
        print(f"  ERROR augmenting split data: {e}")
        return {"model": model_name, "status": "failed", "error": repr(e)}

    if res_split_tr is None:
        print(f"  WARNING: augmentation returned None for split-fit")
        return {"model": model_name, "status": "failed", "error": "augmentation_none"}

    Xtr_aug, Xval_aug, _ = res_split_tr

    est_split = build_regression_model(model_name, extra_params=extra_params)
    est_split.fit(Xtr_aug, y_blend_tr)

    pred_val = clip_pred(est_split.predict(Xval_aug))

    # For split-test predictions, we need to augment test data with blend_train as training context
    try:
        _, Xtest_typ_split_aug, _ = build_source_augmented_data(meta_source, source_config, blend_train, test_typical)
        _, Xtest_full_split_aug, _ = build_source_augmented_data(meta_source, source_config, blend_train, test_full)
    except Exception as e:
        print(f"  ERROR augmenting split-test data: {e}")
        return {"model": model_name, "status": "failed", "error": repr(e)}

    pred_test_typ_split = clip_pred(est_split.predict(Xtest_typ_split_aug))
    pred_test_full_split = clip_pred(est_split.predict(Xtest_full_split_aug))

    # ── 2) Full-fit: train on full train_typical, predict on holdout ──
    try:
        res_full = build_source_augmented_data(meta_source, source_config, train_typical, test_typical)
        _, Xtest_full_fullfit_aug, _ = build_source_augmented_data(meta_source, source_config, train_typical, test_full)
    except Exception as e:
        print(f"  ERROR augmenting full-fit data: {e}")
        return {"model": model_name, "status": "failed", "error": repr(e)}

    if res_full is None:
        return {"model": model_name, "status": "failed", "error": "augmentation_none_fullfit"}

    Xtrain_aug, Xtest_typ_fullfit_aug, _ = res_full

    est_full = build_regression_model(model_name, extra_params=extra_params)
    est_full.fit(Xtrain_aug, ytrain)

    pred_test_typ_fullfit = clip_pred(est_full.predict(Xtest_typ_fullfit_aug))
    pred_test_full_fullfit = clip_pred(est_full.predict(Xtest_full_fullfit_aug))

    # ── Save ──
    _save_pred("split_val", model_name, pred_val)
    _save_pred("split_test_typical", model_name, pred_test_typ_split)
    _save_pred("split_test_full", model_name, pred_test_full_split)
    _save_pred("fullfit_test_typical", model_name, pred_test_typ_fullfit)
    _save_pred("fullfit_test_full", model_name, pred_test_full_fullfit)

    elapsed = (datetime.now() - t0).total_seconds()
    result = {
        "model": model_name,
        "meta_source": meta_source,
        "status": "rebuilt",
        "elapsed_sec": float(elapsed),
        "split_val_MAE": mae(y_blend_val, pred_val),
        "fullfit_typical_MAE": mae(ytest_typical, pred_test_typ_fullfit),
        "fullfit_full_MAE": mae(ytest_full, pred_test_full_fullfit),
    }
    print(f"  split_val_MAE={result['split_val_MAE']:.4f}  fullfit_typ_MAE={result['fullfit_typical_MAE']:.4f}  ({elapsed:.1f}s)")
    return result


# ── Run rebuild for all models ──
rebuild_rows = []
for _, row in meta_df.iterrows():
    try:
        out = rebuild_meta_model(row)
        rebuild_rows.append(out)
    except Exception as e:
        rebuild_rows.append({"model": row["model"], "status": "failed", "error": repr(e)})

rebuild_log_df = pd.DataFrame(rebuild_rows)
display(rebuild_log_df)
rebuild_log_df.to_csv(RUN_DIR / "rebuild_log.csv", index=False)
rebuild_log_df.to_csv(ART_DIR / "rebuild_log_latest.csv", index=False)

# ── Load prediction matrices ──
val_pred_df = load_prediction_frame("split_val", SELECTED_MODELS)
test_typ_fullfit_pred_df = load_prediction_frame("fullfit_test_typical", SELECTED_MODELS)
test_full_fullfit_pred_df = load_prediction_frame("fullfit_test_full", SELECTED_MODELS)

available_models = sorted(
    set(val_pred_df.columns) & set(test_typ_fullfit_pred_df.columns) & set(test_full_fullfit_pred_df.columns),
    key=lambda m: SELECTED_MODELS.index(m) if m in SELECTED_MODELS else 999,
)

val_pred_df = val_pred_df[available_models].copy()
test_typ_fullfit_pred_df = test_typ_fullfit_pred_df[available_models].copy()
test_full_fullfit_pred_df = test_full_fullfit_pred_df[available_models].copy()

print(f"\nAvailable rebuilt meta-tuned models: {len(available_models)}")
print(available_models)

if len(available_models) < 2:
    raise RuntimeError("Need at least 2 successfully rebuilt models for ensembling.")

val_pred_df.to_csv(RUN_DIR / "base_pred_split_val.csv", index=False)
val_pred_df.to_csv(ART_DIR / "base_pred_split_val_latest.csv", index=False)
test_typ_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_typical.csv", index=False)
test_full_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_full.csv", index=False)

y_val = np.asarray(y_blend_val, dtype=float)
y_test_typ = np.asarray(ytest_typical, dtype=float)
y_test_full = np.asarray(ytest_full, dtype=float)

base_leaderboard = get_base_leaderboard_from_predictions(val_pred_df, y_val)
display(base_leaderboard.head(15))
base_leaderboard.to_csv(RUN_DIR / "rebuilt_base_leaderboard.csv", index=False)
base_leaderboard.to_csv(ART_DIR / "rebuilt_base_leaderboard_latest.csv", index=False)


## Ensemble search

Генерация спецификаций ансамблей и оценка на selection-сплите.
Семейства: single, aggregate, weighted, pair_grid, stacker.

In [ ]:
# ── Blend split for ensemble fitting vs selection ──
blend_cut = int(len(y_val) * BLEND_FIT_FRAC)
if blend_cut <= 0 or blend_cut >= len(y_val):
    raise ValueError("Bad BLEND_FIT_FRAC.")

pred_fit_df = val_pred_df.iloc[:blend_cut].copy()
pred_sel_df = val_pred_df.iloc[blend_cut:].copy()
y_fit = y_val[:blend_cut].copy()
y_sel = y_val[blend_cut:].copy()

# ── Rank models by blend-fit MAE ──
fit_rank_df = get_base_leaderboard_from_predictions(pred_fit_df, y_fit)
ranked_models = fit_rank_df["model"].tolist()
print("Top models by blend-fit MAE:")
display(fit_rank_df.head(15))
fit_rank_df.to_csv(RUN_DIR / "blend_fit_model_ranking.csv", index=False)

# ══════════════════════════════════════════════════════════════
#  Generate ensemble specs
# ══════════════════════════════════════════════════════════════
specs = []

# singles
for m in ranked_models:
    specs.append({"family": "single", "name": "single", "models": [m]})

# all-model global aggregations
all_models = ranked_models.copy()
specs.extend([
    {"family": "aggregate", "name": "all_mean", "models": all_models, "agg_method": "mean"},
    {"family": "aggregate", "name": "all_median", "models": all_models, "agg_method": "median"},
    {"family": "aggregate", "name": "all_trim10", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.10},
    {"family": "aggregate", "name": "all_trim20", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.20},
    {"family": "weighted", "name": "all_inverse_mae", "models": all_models, "weight_scheme": "inverse_mae"},
    {"family": "weighted", "name": "all_inverse_mae_sq", "models": all_models, "weight_scheme": "inverse_mae_sq"},
    {"family": "weighted", "name": "all_softmax_t1", "models": all_models, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}},
    {"family": "weighted", "name": "all_nnls", "models": all_models, "weight_scheme": "nnls"},
    {"family": "weighted", "name": "all_simplex_mae", "models": all_models, "weight_scheme": "simplex_mae"},
])

# top-k prefixes
if RUN_TOPK_PREFIX_ENSEMBLES:
    for k in range(2, len(ranked_models) + 1):
        topk = ranked_models[:k]
        specs.append({"family": "aggregate", "name": f"top{k}_mean", "models": topk, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": f"top{k}_median", "models": topk, "agg_method": "median"})
        if k >= 5:
            specs.append({"family": "aggregate", "name": f"top{k}_trim10", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.10})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae", "models": topk, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae_sq", "models": topk, "weight_scheme": "inverse_mae_sq"})
        for temp in [0.25, 0.5, 1.0, 2.0, 4.0]:
            specs.append({"family": "weighted", "name": f"top{k}_softmax_t{temp}", "models": topk, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": temp}})
        for power in [0.5, 1.0, 2.0]:
            specs.append({"family": "weighted", "name": f"top{k}_rankpow_{power}", "models": topk, "weight_scheme": "rank_power", "weight_kwargs": {"power": power}})
        specs.append({"family": "weighted", "name": f"top{k}_nnls", "models": topk, "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": f"top{k}_simplex_mae", "models": topk, "weight_scheme": "simplex_mae"})

# all pairs
if RUN_ALL_PAIRS:
    for a, b in combinations(ranked_models, 2):
        specs.append({"family": "aggregate", "name": "pair_mean", "models": [a, b], "agg_method": "mean"})
        specs.append({"family": "pair_grid", "name": "pair_grid", "models": [a, b]})
        specs.append({"family": "weighted", "name": "pair_nnls", "models": [a, b], "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": "pair_simplex", "models": [a, b], "weight_scheme": "simplex_mae"})

# all triples
if RUN_ALL_TRIPLES:
    for trio in combinations(ranked_models, 3):
        trio = list(trio)
        specs.append({"family": "aggregate", "name": "triple_mean", "models": trio, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": "triple_median", "models": trio, "agg_method": "median"})
        specs.append({"family": "weighted", "name": "triple_inverse", "models": trio, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": "triple_nnls", "models": trio, "weight_scheme": "nnls"})

# exhaustive subsets on top-N
if RUN_EXHAUSTIVE_TOPN_SUBSETS:
    topn_models = ranked_models[:min(EXHAUSTIVE_TOP_N, len(ranked_models))]
    for r in range(2, len(topn_models) + 1):
        for subset in combinations(topn_models, r):
            subset = list(subset)
            specs.append({"family": "aggregate", "name": f"subset{r}_mean", "models": subset, "agg_method": "mean"})
            specs.append({"family": "aggregate", "name": f"subset{r}_median", "models": subset, "agg_method": "median"})
            if r >= 5:
                specs.append({"family": "aggregate", "name": f"subset{r}_trim10", "models": subset, "agg_method": "trimmed_mean", "trim_frac": 0.10})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse", "models": subset, "weight_scheme": "inverse_mae"})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse_sq", "models": subset, "weight_scheme": "inverse_mae_sq"})
            specs.append({"family": "weighted", "name": f"subset{r}_softmax", "models": subset, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}})
            specs.append({"family": "weighted", "name": f"subset{r}_nnls", "models": subset, "weight_scheme": "nnls"})
            specs.append({"family": "weighted", "name": f"subset{r}_simplex", "models": subset, "weight_scheme": "simplex_mae"})

# greedy forward search
if RUN_GREEDY_SEARCH:
    specs.extend(greedy_forward_specs(ranked_models))

# stackers
if RUN_STACKERS:
    stackers = ["linear", "positive_linear", "ridge", "lasso", "elastic", "bayes", "huber", "quantile", "gbr", "rf", "etr"]
    if HAS_XGB:
        stackers.append("xgb")
    for stacker_name in stackers:
        specs.append({"family": "stacker", "name": f"all_{stacker_name}", "models": all_models, "stacker_name": stacker_name})
        for k in range(2, len(ranked_models) + 1):
            topk = ranked_models[:k]
            specs.append({"family": "stacker", "name": f"top{k}_{stacker_name}", "models": topk, "stacker_name": stacker_name})

print(f"Total ensemble specs: {len(specs)}")

# ══════════════════════════════════════════════════════════════
#  Search on selection split
# ══════════════════════════════════════════════════════════════
search_rows = []
for idx, spec in enumerate(specs, start=1):
    if idx % 200 == 0:
        print(f"Selection scoring {idx}/{len(specs)} ...")
    try:
        row = evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
        search_rows.append(row)
    except Exception as e:
        search_rows.append({
            "tag": spec_tag(spec),
            "family": spec["family"],
            "name": spec["name"],
            "n_models": len(spec["models"]),
            "models": spec["models"],
            "selection_MAE": np.inf,
            "selection_RMSE": np.inf,
            "selection_MdAE": np.inf,
            "error": repr(e),
            "spec": spec,
        })

search_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "error": r.get("error", ""),
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in search_rows
]).sort_values(["selection_MAE", "selection_RMSE", "selection_MdAE"], kind="stable").reset_index(drop=True)

display(search_df.head(30))
search_df.to_csv(RUN_DIR / "ensemble_search_leaderboard.csv", index=False)
search_df.to_csv(ART_DIR / "ensemble_search_leaderboard_latest.csv", index=False)

top_refit_rows = [r for r in search_rows if np.isfinite(r["selection_MAE"])][:min(REFIT_TOP_ENSEMBLES, len(search_rows))]
print(f"Top rows selected for holdout refit: {len(top_refit_rows)}")


## Refit top ensembles on holdout

Лучшие ансамбли по selection-MAE рефитим на полном validation и оцениваем на holdout.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Refit top ensembles → holdout evaluation
# ══════════════════════════════════════════════════════════════
final_rows = []
for idx, row in enumerate(top_refit_rows, start=1):
    if idx % 25 == 0:
        print(f"Final holdout refit {idx}/{len(top_refit_rows)} ...")
    spec = row["spec"]
    out = refit_and_eval_spec(
        spec,
        pred_val_df=val_pred_df,
        y_val=y_val,
        pred_test_typ_df=test_typ_fullfit_pred_df,
        y_test_typ=y_test_typ,
        pred_test_full_df=test_full_fullfit_pred_df,
        y_test_full=y_test_full,
    )
    out["selection_MAE"] = row["selection_MAE"]
    out["selection_RMSE"] = row["selection_RMSE"]
    out["selection_MdAE"] = row["selection_MdAE"]
    final_rows.append(out)

final_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "MAE_typical": r["MAE_typical"],
        "RMSE_typical": r["RMSE_typical"],
        "MdAE_typical": r["MdAE_typical"],
        "MAE_full": r["MAE_full"],
        "RMSE_full": r["RMSE_full"],
        "MdAE_full": r["MdAE_full"],
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weights_like": json.dumps(r.get("weights_like", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in final_rows
]).sort_values(["selection_MAE", "MAE_typical", "MAE_full"], kind="stable").reset_index(drop=True)

display(final_df.head(30))
final_df.to_csv(RUN_DIR / "ensemble_holdout_leaderboard.csv", index=False)
final_df.to_csv(ART_DIR / "ensemble_holdout_leaderboard_latest.csv", index=False)

# ── Best ensemble summary ──
best_ensemble_row = final_rows[0]
best_single_model = min(available_models, key=lambda m: mae(y_val, val_pred_df[m].values))

single_typ = calc_reg_metrics(y_test_typ, test_typ_fullfit_pred_df[best_single_model].values)
single_full = calc_reg_metrics(y_test_full, test_full_fullfit_pred_df[best_single_model].values)

comparison_payload = {
    "selection_metric": "blend_select_MAE",
    "blend_fit_frac": BLEND_FIT_FRAC,
    "available_models": available_models,
    "best_single_model_by_val": best_single_model,
    "best_single_model_metrics": {
        "MAE_typical": single_typ["MAE"],
        "RMSE_typical": single_typ["RMSE"],
        "MdAE_typical": single_typ["MdAE"],
        "MAE_full": single_full["MAE"],
        "RMSE_full": single_full["RMSE"],
        "MdAE_full": single_full["MdAE"],
    },
    "best_ensemble": {
        "tag": best_ensemble_row["tag"],
        "family": best_ensemble_row["family"],
        "name": best_ensemble_row["name"],
        "models": best_ensemble_row["models"],
        "selection_MAE": best_ensemble_row["selection_MAE"],
        "MAE_typical": best_ensemble_row["MAE_typical"],
        "RMSE_typical": best_ensemble_row["RMSE_typical"],
        "MdAE_typical": best_ensemble_row["MdAE_typical"],
        "MAE_full": best_ensemble_row["MAE_full"],
        "RMSE_full": best_ensemble_row["RMSE_full"],
        "MdAE_full": best_ensemble_row["MdAE_full"],
        "weights": best_ensemble_row.get("weights", []),
        "weights_like": best_ensemble_row.get("weights_like", []),
        "weight_first": best_ensemble_row.get("weight_first", None),
    },
}

display(pd.DataFrame([
    {"item": "best_single_model_by_val", "value": best_single_model},
    {"item": "best_ensemble_tag", "value": best_ensemble_row["tag"]},
    {"item": "best_ensemble_selection_MAE", "value": best_ensemble_row["selection_MAE"]},
    {"item": "best_ensemble_MAE_typical", "value": best_ensemble_row["MAE_typical"]},
    {"item": "best_ensemble_MAE_full", "value": best_ensemble_row["MAE_full"]},
]))

with open(RUN_DIR / "best_ensemble_summary.json", "w", encoding="utf-8") as f:
    json.dump(json_ready(comparison_payload), f, ensure_ascii=False, indent=2)
with open(ART_DIR / "best_ensemble_summary_latest.json", "w", encoding="utf-8") as f:
    json.dump(json_ready(comparison_payload), f, ensure_ascii=False, indent=2)

best_models = best_ensemble_row["models"]
best_weights = best_ensemble_row.get("weights") or best_ensemble_row.get("weights_like") or []
if len(best_weights) == len(best_models) and len(best_weights) > 0:
    wdf = pd.DataFrame({"model": best_models, "weight": best_weights})
    wdf.to_csv(RUN_DIR / "best_ensemble_weights.csv", index=False)
    wdf.to_csv(ART_DIR / "best_ensemble_weights_latest.csv", index=False)

# ── Run manifest ──
run_manifest = {
    "run_name": RUN_NAME,
    "data_path": str(DATA_PATH),
    "meta_summary_path": str(META_SUMMARY_PATH),
    "artifacts_root": str(ART_DIR),
    "run_dir": str(RUN_DIR),
    "duration_cap": DURATION_CAP,
    "train_full_rows": int(len(train_full)),
    "train_typical_rows": int(len(train_typical)),
    "inner_blend_train_rows": int(len(blend_train)),
    "inner_blend_val_rows": int(len(blend_val)),
    "holdout_typical_rows": int(len(test_typical)),
    "holdout_full_rows": int(len(test_full)),
    "n_available_models": len(available_models),
    "available_models": available_models,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "total_specs_tested": len(specs),
    "top_refit_count": len(top_refit_rows),
}
with open(RUN_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(json_ready(run_manifest), f, ensure_ascii=False, indent=2)

print("\nExperiment complete.")
print(f"Best single: {best_single_model} -> MAE_typ={single_typ['MAE']:.4f}")
print(f"Best ensemble: {best_ensemble_row['tag']} -> MAE_typ={best_ensemble_row['MAE_typical']:.4f}")


## Quick plots / diagnostics

In [ ]:
# ── Top-15 ensembles bar plot ──
top_plot = final_df.head(15).copy()
plt.figure(figsize=(12, 6))
plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1])
plt.xlabel("MAE_typical")
plt.ylabel("Ensemble")
plt.title("Top-15 Meta-ML ensembles by selection_MAE -> holdout typical")
plt.tight_layout()
plt.show()

# ── Correlation of validation predictions ──
top_corr_models = ranked_models[:min(7, len(ranked_models))]
corr = val_pred_df[top_corr_models].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation of validation predictions (meta-tuned ML pool)")
plt.tight_layout()
plt.show()

# ── Near-ties ──
display(final_df.head(20))
near_ties = final_df[final_df["selection_MAE"] <= final_df["selection_MAE"].min() + 0.5].copy()
display(near_ties.head(20))


## Что смотреть после запуска

Главные артефакты:
- `ensemble_search_leaderboard.csv`
- `ensemble_holdout_leaderboard.csv`
- `best_ensemble_summary.json`
- `best_ensemble_weights.csv` (если есть)
- `run_manifest.json`
- `rebuild_log.csv`

Все лежат в `artifacts_meta_ensembles_final/<run_name>/`